# Finetuning des TrOCR-large-handwriting Models mit dem MathWriting Dataset:

- [Quelle Finetuning](https://github.com/NielsRogge/Transformers-Tutorials/blob/master/TrOCR/Fine_tune_TrOCR_on_IAM_Handwriting_Database_using_Seq2SeqTrainer.ipynb)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 
os.environ["WANDB_DISABLED"] = "true"

In [2]:
!pip install transformers==4.45.2 sentence-transformers==3.1.1
#!pip install kagglehub

## Datensatz laden

In [3]:
from datasets import load_dataset

ds = load_dataset("deepcopy/MathWriting-Human")

train_data = [(s["image"], s["latex"]) for s in ds["train"]]
val_data   = [(s["image"], s["latex"]) for s in ds["val"]]
test_data  = [(s["image"], s["latex"]) for s in ds["test"]]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

Train: 229864, Val: 15674, Test: 7644


In [4]:
#https://www.geeksforgeeks.org/python/how-to-uncompress-a-tar-gz-file-using-python/
#import tarfile

#file = tarfile.open('mathwriting-2024.tgz') 

# Entpacken in Zielordner
#file.extractall('./mathwriting_data') 
#file.close()

## Datensatzklasse

In [5]:
from torch.utils.data import Dataset
from PIL import Image
import torch

class MathWritingDataset(Dataset):
    def __init__(self, data, processor, max_target_length=128):
        self.data = data
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.data) # Wird für die Batch-Verarbeitung gebraucht (DataLoader)

    # Zugriff auf ein einzelnes Sample:
    def __getitem__(self, idx):
        image, latex = self.data[idx]
        image = image.convert("RGB")

        pixel_values = self.processor(image, return_tensors="pt").pixel_values #https://stackoverflow.com/questions/75891072/valueerror-unable-to-infer-channel-dimension-format
        labels = self.processor.tokenizer(text_target=latex ,padding="max_length",truncation=True,max_length=self.max_target_length).input_ids  # input_ids: Liste der Token-Zahlen
        # Padding Tokens werden hier ignoriert: <PAD> Tokens werden bei zu kurzen Texten als Auffüll-Zeichen genutzt, der Cross-Entropy Loss soll diese ignorieren (geschieht durch setzen von -100)
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels] 

        encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)} # encoding hat: pixel_values (Tensor des Bildes), labels (Tensor der Token-IDs des Textes)
        return encoding

In [6]:
from transformers import TrOCRProcessor
from torch.utils.data import Subset
import random

processor_fhswf = TrOCRProcessor.from_pretrained("fhswf/TrOCR_Math_handwritten")
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-large-stage1')
train_dataset = MathWritingDataset(train_data, processor)
val_dataset   = MathWritingDataset(val_data,processor)
test_dataset  = MathWritingDataset(test_data, processor)
test_dataset_fhswf  = MathWritingDataset(test_data, processor)

## Modell laden

In [7]:
from transformers import VisionEncoderDecoderModel
import torch
import inspect

#https://huggingface.co/fhswf/TrOCR_Math_handwritten

model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-large-stage1")
#model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-large-handwritten")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(inspect.getsource(VisionEncoderDecoderModel.forward))

VisionEncoderDecoderModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-large-stage1 and are newly initialized: ['enco

    @add_start_docstrings_to_model_forward(VISION_ENCODER_DECODER_INPUTS_DOCSTRING)
    @replace_return_docstrings(output_type=Seq2SeqLMOutput, config_class=_CONFIG_FOR_DOC)
    def forward(
        self,
        pixel_values: Optional[torch.FloatTensor] = None,
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.BoolTensor] = None,
        encoder_outputs: Optional[Tuple[torch.FloatTensor]] = None,
        past_key_values: Optional[Tuple[Tuple[torch.FloatTensor]]] = None,
        decoder_inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        **kwargs,
    ) -> Union[Tuple[torch.FloatTensor], Seq2SeqLMOutput]:
        r"""
        Returns:

        Examples:

        ```python
        >>> f

## Konfigurationen von den Special-Tokens und Beam-Search für die Textgenerierung

In [8]:
# interessante Quelle (zum nachrecherchieren): https://huggingface.co/docs/transformers/main/model_doc/trocr   und https://huggingface.co/docs/transformers/main_classes/text_generation
# set special tokens used for creating the decoder_input_ids from the labels
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id #Normalerweise ist Starttoken <BOS> hier aber <CLS>
model.config.pad_token_id = processor.tokenizer.pad_token_id 
# make sure vocab size is set correctly
model.config.vocab_size = model.config.decoder.vocab_size


# set beam search parameters
model.config.eos_token_id = processor.tokenizer.sep_token_id

In [9]:
from transformers import GenerationConfig

model.generation_config = GenerationConfig(
    decoder_start_token_id=processor.tokenizer.cls_token_id,
    pad_token_id=processor.tokenizer.pad_token_id,
    eos_token_id=processor.tokenizer.sep_token_id,
    max_length=128,
    num_beams=4,
    early_stopping=True,
    length_penalty=1.0
)



## Metriken laden

In [10]:
!pip install editdistance

In [11]:
from evaluate import load
#import editdistance
cer = load("cer")


In [12]:
import editdistance
def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    
    cer_score = cer.compute(predictions=pred_str, references=label_str)
    
    # Edit Distance Schwellenwert-Metriken
    edit_distances = [
        editdistance.eval(p, r)
        for p, r in zip(pred_str, label_str)
    ]
    n = len(edit_distances)
    exact_match      = sum(d == 0 for d in edit_distances) / n
    one_error_match  = sum(d <= 1 for d in edit_distances) / n
    two_error_match  = sum(d <= 2 for d in edit_distances) / n
    
    return {
        "cer": cer_score,
        "exact_match":     exact_match,      
        "one_error_match": one_error_match,  
        "two_error_match": two_error_match, 
    }

## Hyperparameter setzen

In [13]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
# https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer#transformers.Seq2SeqTrainingArguments
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    evaluation_strategy="epoch",
    per_device_train_batch_size=32, # Batchgröße pro CPU/GPU
    per_device_eval_batch_size=32,
    bf16=True,  # Teile des Trainings laufen mit 16 Bit (spart Speicher und beschleunigt das Training)
    output_dir="./model-mathwriting_large_stage_mw", #Verzeichnis in dem Modelle etc. gespeichert werden_
    logging_steps=100,
    save_strategy="epoch",
    save_steps=3000,
    seed=42,
    num_train_epochs=4,
    gradient_accumulation_steps=2,
    warmup_ratio= 0.1,
    learning_rate=3.5e-5
)

/opt/conda/lib/python3.11/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [15]:
print(training_args.generation_max_length, training_args.warmup_ratio)

None 0.1


## Beginn des Trainings

In [16]:
from transformers import default_data_collator
torch.cuda.empty_cache()
# https://huggingface.co/docs/transformers/main_classes/trainer

trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.tokenizer,
    args=training_args, # Sind oben definiert
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    #eval_dataset=small_eval_dataset,
    data_collator=default_data_collator,  # collator: Erstellt Batches aus den Elementen der Trainings oder Validierungsdaten
)

trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sandzo (sandzo-fh-swf) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Cer,Exact Match,One Error Match,Two Error Match
1,0.148600,0.182234,0.099285,0.558249,0.677619,0.727766
2,0.079200,0.126593,0.071264,0.664476,0.761580,0.800242
3,0.041300,0.108925,0.061536,0.695993,0.787036,0.824869
4,0.018400,0.103498,0.055854,0.721258,0.806048,0.839862


TrainOutput(global_step=14368, training_loss=0.17802796454764155, metrics={'train_runtime': 31006.1835, 'train_samples_per_second': 29.654, 'train_steps_per_second': 0.463, 'total_flos': 1.3610216367775065e+21, 'train_loss': 0.17802796454764155, 'epoch': 4.0})

## Modell mit Mathwriting:

In [17]:
from transformers import default_data_collator
torch.cuda.empty_cache()
# https://huggingface.co/docs/transformers/main_classes/trainer

trained_model = VisionEncoderDecoderModel.from_pretrained('model-mathwriting_large_stage_mw/checkpoint-14368/').to(device)

t_trainer = Seq2SeqTrainer(
    model=trained_model,
    tokenizer=processor.tokenizer,
    args=training_args, # Sind oben definiert
    compute_metrics=compute_metrics,
    #train_dataset=train_dataset,
    #eval_dataset=val_dataset,
    eval_dataset=test_dataset,
    #eval_dataset=small_eval_dataset,
    data_collator=default_data_collator  # collator: Erstellt Batches aus den Elementen der Trainings oder Validierungsdaten
)

In [18]:
test_res = t_trainer.evaluate()

print(test_res)

{'eval_loss': 0.13808032870292664, 'eval_model_preparation_time': 0.0074, 'eval_cer': 0.07727811296247639, 'eval_exact_match': 0.6045264259549974, 'eval_one_error_match': 0.7110151753008895, 'eval_two_error_match': 0.76007326007326, 'eval_runtime': 747.2588, 'eval_samples_per_second': 10.229, 'eval_steps_per_second': 0.32}


## Modell mit HME+Mathwriting evaluieren:

In [26]:
from transformers import default_data_collator, TrOCRProcessor, VisionEncoderDecoderModel
import os

model_path = "/home/dedol002/math_trocr_hme_and_mathwriting"

torch.cuda.empty_cache()

processor = TrOCRProcessor.from_pretrained(model_path)
trained_model = VisionEncoderDecoderModel.from_pretrained(model_path).to(device)

t_trainer = Seq2SeqTrainer(
    model=trained_model,
    tokenizer=processor.tokenizer,
    args=training_args,
    compute_metrics=compute_metrics,
    eval_dataset=test_dataset,
    data_collator=default_data_collator
)


In [27]:
res = t_trainer.evaluate()

print(res)

{'eval_loss': 0.13240718841552734, 'eval_model_preparation_time': 0.0173, 'eval_cer': 0.07506209317198198, 'eval_exact_match': 0.6170852956567242, 'eval_one_error_match': 0.7231815803244375, 'eval_two_error_match': 0.77145473574045, 'eval_runtime': 744.4728, 'eval_samples_per_second': 10.268, 'eval_steps_per_second': 0.321}


## Modell untrainiert:

In [30]:
processor_base = TrOCRProcessor.from_pretrained('microsoft/trocr-large-stage1')
base_model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-large-stage1').to(device)

# Gleiche Configs wie beim Training
base_model.config.decoder_start_token_id = processor_base.tokenizer.cls_token_id
base_model.config.pad_token_id = processor_base.tokenizer.pad_token_id
base_model.config.vocab_size = base_model.config.decoder.vocab_size
base_model.config.eos_token_id = processor_base.tokenizer.sep_token_id

from transformers import GenerationConfig
base_model.generation_config = GenerationConfig(
    decoder_start_token_id=processor_base.tokenizer.cls_token_id,
    pad_token_id=processor_base.tokenizer.pad_token_id,
    eos_token_id=processor_base.tokenizer.sep_token_id,
    max_length=128,
    num_beams=4,
    early_stopping=True,
    length_penalty=1.0
)

base_trainer = Seq2SeqTrainer(
    model=base_model,
    tokenizer=processor_base.tokenizer,
    args=training_args,
    compute_metrics=compute_metrics,
    eval_dataset=test_dataset,
    data_collator=default_data_collator
)

base_res = base_trainer.evaluate()
print(base_res)

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-large-stage1 and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'eval_loss': 11.584779739379883, 'eval_model_preparation_time': 0.0076, 'eval_cer': 0.9902336487980448, 'eval_exact_match': 0.0, 'eval_one_error_match': 0.0020931449502878076, 'eval_two_error_match': 0.004447933019361591, 'eval_runtime': 440.0414, 'eval_samples_per_second': 17.371, 'eval_steps_per_second': 0.543}


## FH SWF Modell:

In [14]:
import editdistance
def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor_fhswf.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor_fhswf.tokenizer.pad_token_id
    label_str = processor_fhswf.batch_decode(labels_ids, skip_special_tokens=True)
    
    cer_score = cer.compute(predictions=pred_str, references=label_str)
    
    # Edit Distance Schwellenwert-Metriken
    edit_distances = [
        editdistance.eval(p, r)
        for p, r in zip(pred_str, label_str)
    ]
    n = len(edit_distances)
    exact_match      = sum(d == 0 for d in edit_distances) / n
    one_error_match  = sum(d <= 1 for d in edit_distances) / n
    two_error_match  = sum(d <= 2 for d in edit_distances) / n
    
    return {
        "cer": cer_score,
        "exact_match":     exact_match,      
        "one_error_match": one_error_match,  
        "two_error_match": two_error_match, 
    }

In [15]:
from transformers import default_data_collator
from transformers import VisionEncoderDecoderModel
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

torch.cuda.empty_cache()
# https://huggingface.co/docs/transformers/main_classes/trainer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fhswf_model = VisionEncoderDecoderModel.from_pretrained("fhswf/TrOCR_Math_handwritten").to(device)
processor_fhswf = TrOCRProcessor.from_pretrained("fhswf/TrOCR_Math_handwritten")

fhswf_trainer = Seq2SeqTrainer(
    model=fhswf_model,
    tokenizer=processor_fhswf.tokenizer,
    args=training_args, # Sind oben definiert
    compute_metrics=compute_metrics,
    #train_dataset=train_dataset,
    #eval_dataset=val_dataset,
    eval_dataset=test_dataset_fhswf,
    #eval_dataset=small_eval_dataset,
    data_collator=default_data_collator  # collator: Erstellt Batches aus den Elementen der Trainings oder Validierungsdaten
)

In [16]:
fhswf_res = fhswf_trainer.evaluate()
print(fhswf_res)

{'eval_loss': 0.2485220730304718, 'eval_model_preparation_time': 0.0082, 'eval_cer': 0.15132886824349448, 'eval_exact_match': 0.4314495028780743, 'eval_one_error_match': 0.5484039769754055, 'eval_two_error_match': 0.6049188906331764, 'eval_runtime': 476.4496, 'eval_samples_per_second': 16.044, 'eval_steps_per_second': 0.502}
